# Day 5: Hyperparameter Tuning and Cross-Validation

## 1. Navigating the Hyperparameter Phase Space
Yesterday, we achieved a strong ROC-AUC of 0.9047 using XGBoost. However, we used "Default" settings. In high-energy physics, we must optimize the model's "knobs" (hyperparameters) to find the absolute global minimum of the Loss Function.

### The Problem: Bias-Variance Trade-off
- **Low Model Complexity:** High Bias (Underfitting the Higgs signal).
- **High Model Complexity:** High Variance (Overfitting to detector noise).

### The Solution: GridSearchCV & Stratified K-Fold
We use **Grid Search** to perform a brute-force scan across a specified range of hyperparameters. To ensure the results are robust and not a statistical fluke of a single data split, we use **K-Fold Cross-Validation**.



**Stratified K-Fold** is critical here because our dataset is imbalanced (65% Background, 35% Signal). Stratification ensures that every "fold" maintains this exact ratio, preventing the model from training on a biased subset of the physics.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('training.csv').replace(-999.0, np.nan)

cols_to_drop = df.columns[df.isna().mean() > 0.70].tolist() + ['EventId', 'Weight']
df_clean = df.drop(columns=cols_to_drop)

for col in df_clean.columns:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean['Label'] = df_clean['Label'].map({'s': 1, 'b': 0})
X = df_clean.drop(columns=['Label'])
y = df_clean['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data successfully loaded into memory. Ready for Optimization.")

Data successfully loaded into memory. Ready for Optimization.


In [3]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

print("--- Initializing Systematic Grid Search ---")

xgb_model = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss', 
    tree_method='hist',
    random_state=42
)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200]
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    verbose=2, 
    n_jobs=-1  
)

print("Searching for optimal parameters... (This may take 3-5 minutes)")
grid_search.fit(X_train, y_train)

print("\n--- Optimization Complete ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validated ROC-AUC: {grid_search.best_score_:.4f}")

--- Initializing Systematic Grid Search ---
Searching for optimal parameters... (This may take 3-5 minutes)
Fitting 3 folds for each of 18 candidates, totalling 54 fits


c:\Users\harsh\OneDrive\Desktop\higgs_boson\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:14:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Optimization Complete ---
Best Parameters Found: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
Best Cross-Validated ROC-AUC: 0.9066


## 3. Training the Optimized Production Model
Now that `GridSearchCV` has identified the global minimum in the parameter space, we instantiate the final version of our XGBoost classifier. We will evaluate this on the 20% 'Test Set' that the grid search never saw to ensure our 0.9066 AUC is generalizable.

In [4]:
from sklearn.metrics import classification_report, roc_auc_score

print("--- Training Final Optimized Model ---")

final_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

final_xgb.fit(X_train, y_train)

y_pred_final = final_xgb.predict(X_test)
y_prob_final = final_xgb.predict_proba(X_test)[:, 1]

final_auc = roc_auc_score(y_test, y_prob_final)
print(f"\nFINAL TEST ROC-AUC: {final_auc:.4f}")
print("\nFinal Physics Classification Report:")
print(classification_report(y_test, y_pred_final))

--- Training Final Optimized Model ---


c:\Users\harsh\OneDrive\Desktop\higgs_boson\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:15:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



FINAL TEST ROC-AUC: 0.9061

Final Physics Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88     32867
           1       0.78      0.73      0.75     17133

    accuracy                           0.84     50000
   macro avg       0.82      0.81      0.82     50000
weighted avg       0.84      0.84      0.84     50000



In [5]:
import os

if not os.path.exists('models'):
    os.makedirs('models')

final_xgb.save_model('models/higgs_classifier_v1.json')

print("Final production model saved to 'models/higgs_classifier_v1.json'")

Final production model saved to 'models/higgs_classifier_v1.json'
